In [0]:
from pyspark.sql import functions as F, Window

CATALOG, Y = "workspace", 2025
BRONZE = f"{CATALOG}.rc_bronze.survey_results"
SILVER = f"{CATALOG}.rc_silver.survey_clean"
LOW_EFFORT_MAX = 5
SALARY_MIN, SALARY_MAX = 1_000, 1_000_000   # hypothèse d'analyste, pas une règle de la source
INT_COLS = ["WorkExp", "YearsCode", "JobSat"]
PRO_LABEL = "I am a developer by profession"

b = spark.table(BRONZE).where(f"_survey_year = {Y}")
src_cols = [r.bronze_name for r in spark.table(f"{CATALOG}.rc_bronze.column_map")
            .where(f"survey_year = {Y}").orderBy("ordinal").collect()]
content_cols = [c for c in src_cols if c != "ResponseId"]

aggs = []
for c in src_cols:
    aggs += [F.sum((F.col(c) != "NA").cast("int")).alias(f"{c}__n"),
             F.sum(F.expr(f"try_cast(`{c}` as double) is not null").cast("int")).alias(f"{c}__num")]
r = b.agg(*aggs).collect()[0].asDict()
numeric_cols = [c for c in src_cols if r[f"{c}__n"] > 0 and r[f"{c}__n"] == r[f"{c}__num"]]
print(f"{len(numeric_cols)} colonnes entièrement numériques (53 attendues)")

missing = [c for c in INT_COLS + ["ConvertedCompYearly", "MainBranch"] if c not in src_cols]
assert not missing, f"CRITIQUE : colonnes absentes : {missing}"
assert all(c in numeric_cols for c in INT_COLS + ["ConvertedCompYearly"]), "CRITIQUE : colonne attendue non numérique"
for c in INT_COLS:
    non_int = b.where(f"`{c}` != 'NA' and try_cast(`{c}` as double) != floor(try_cast(`{c}` as double))").count()
    assert non_int == 0, f"CRITIQUE : {c} contient {non_int} valeurs non entières"
print("Contrôles préalables OK")

53 colonnes entièrement numériques (53 attendues)
Contrôles préalables OK


In [0]:
sel = []
for c in src_cols:
    if c == "ResponseId":
        expr = "cast(trim(`ResponseId`) as bigint)"
    elif c in numeric_cols:
        inner = f"try_cast(nullif(trim(`{c}`), 'NA') as double)"
        expr = f"cast({inner} as int)" if c in INT_COLS else inner
    else:
        expr = f"nullif(nullif(trim(`{c}`), 'NA'), '')"
    sel.append(F.expr(expr).alias(c))

s0 = b.select(*sel, F.col("_survey_year"), F.col("_batch_id").alias("_bronze_batch_id"))

answered = sum(F.col(c).isNotNull().cast("int") for c in content_cols)
row_hash = F.sha2(F.concat_ws("\u0001", *[F.coalesce(F.col(c).cast("string"), F.lit("\u0002"))
                                          for c in content_cols]), 256)
w_grp = Window.partitionBy("_row_hash")
w_rank = Window.partitionBy("_row_hash").orderBy("ResponseId")

s1 = (s0.withColumn("n_answered", answered)
        .withColumn("_row_hash", row_hash)
        .withColumn("dup_group_size", F.count("*").over(w_grp))
        .withColumn("_dup_rank", F.row_number().over(w_rank))
        .withColumn("is_low_effort", F.col("n_answered") <= LOW_EFFORT_MAX)
        .withColumn("is_content_duplicate", F.col("_dup_rank") > 1)
        .withColumn("is_professional", F.col("MainBranch") == PRO_LABEL)
        .withColumn("is_salary_extreme",
                    F.when(F.col("ConvertedCompYearly").isNull(), F.lit(None).cast("boolean"))
                     .otherwise((F.col("ConvertedCompYearly") < SALARY_MIN) |
                                (F.col("ConvertedCompYearly") > SALARY_MAX)))
        .withColumn("_silver_processed_at", F.current_timestamp())
        .drop("_dup_rank", "_row_hash"))

(s1.write.format("delta").mode("overwrite")
   .option("replaceWhere", f"_survey_year = {Y}")
   .saveAsTable(SILVER))
print("Écriture terminée :", SILVER)

Écriture terminée : workspace.rc_silver.survey_clean


In [0]:
s = spark.table(SILVER).where(f"_survey_year = {Y}")
n_b, n_s = b.count(), s.count()
print(f"Lignes : Bronze {n_b:,} | Silver {n_s:,}")
assert n_b == n_s, "CRITIQUE : nombre de lignes différent"
assert s.select("ResponseId").distinct().count() == n_s and s.where("ResponseId is null").count() == 0, \
    "CRITIQUE : ResponseId nul ou dupliqué"

bz = b.agg(*[F.sum((F.col(c) == "NA").cast("int")).alias(c) for c in src_cols]).collect()[0].asDict()
sv = s.agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in src_cols]).collect()[0].asDict()
bad = [(c, bz[c], sv[c]) for c in src_cols if (bz[c] or 0) != (sv[c] or 0)]
print("Colonnes dont les NULL Silver ≠ les 'NA' Bronze :", bad)
assert not bad, "CRITIQUE : réconciliation des manquants"

str_cols = [c for c, t in s.dtypes if t == "string" and c in src_cols]
na_left = s.agg(*[F.sum((F.col(c) == "NA").cast("int")).alias(c) for c in str_cols]).collect()[0].asDict()
print("Chaînes 'NA' restantes :", {c: v for c, v in na_left.items() if v})

print("\nIndicateurs (valeurs attendues d'après l'EDA entre parenthèses) :")
s.agg(F.sum(F.col("is_low_effort").cast("int")).alias("low_effort (2015)"),
      F.sum(F.col("is_content_duplicate").cast("int")).alias("duplicats en excès (1547)"),
      F.sum(F.col("is_professional").cast("int")).alias("professionnels (37467)"),
      F.sum(F.col("is_salary_extreme").cast("int")).alias("salaires extrêmes (770)"),
      F.sum(F.col("ConvertedCompYearly").isNotNull().cast("int")).alias("salaires renseignés (23947)")
      ).show(truncate=False)
print(dict(s.select("ResponseId", "WorkExp", "YearsCode", "JobSat", "ConvertedCompYearly", "is_low_effort").dtypes))
spark.sql(f"DESCRIBE DETAIL {SILVER}").select("format", "numFiles", "sizeInBytes").show(truncate=False)

Lignes : Bronze 49,191 | Silver 49,191
Colonnes dont les NULL Silver ≠ les 'NA' Bronze : []
Chaînes 'NA' restantes : {}

Indicateurs (valeurs attendues d'après l'EDA entre parenthèses) :
+-----------------+-------------------------+----------------------+-----------------------+---------------------------+
|low_effort (2015)|duplicats en excès (1547)|professionnels (37467)|salaires extrêmes (770)|salaires renseignés (23947)|
+-----------------+-------------------------+----------------------+-----------------------+---------------------------+
|2015             |1547                     |37467                 |770                    |23947                      |
+-----------------+-------------------------+----------------------+-----------------------+---------------------------+

{'ResponseId': 'bigint', 'WorkExp': 'int', 'YearsCode': 'int', 'JobSat': 'int', 'ConvertedCompYearly': 'double', 'is_low_effort': 'boolean'}
+------+--------+-----------+
|format|numFiles|sizeInBytes|
+-----